
# 통일 점진 규칙: 하나의 판정 방식을 세 문제에 똑같이

지금까지는 문제마다 판정 규칙이 달랐습니다.
- **EPL 규칙 개선:** 빠른 점진법 (스코어 신호 + 신뢰 시퀀스 + 안전장치)
- **혼인·청년 순이동 경보:** CUSUM + 실무 크기 조건

둘 다 **"증거의 하한이 실무 최소 크기를 넘으면 판정한다"**의 변형입니다. 이것을 하나로 만들고, 세 문제에 똑같이 적용해 기존 맞춤 규칙과 비교합니다.

## 통일 점진 규칙
1. **한 번에 하나:** 후보는 현재 규칙에서 하나만 바꾼 것. 경보에서는 "증가 / 감소" 두 방향.
2. **가장 정보가 많은 신호:** 같은 시점에 얻을 수 있는 가장 촘촘한 신호(축구는 스코어, 경보는 월별 자료).
3. **판정:** 언제 멈춰도 유효한 하한(정규 혼합 신뢰 시퀀스)이 실무 최소 크기 m을 넘으면 판정. 오판 허용은 후보 수(방향 수)로 나눔.
4. **안전장치:** 최종 결과 지표가 나빠지는 중이면 보류.
5. **기준값을 정하는 규칙도 하나:** 보정 기간에서 정하고, 채점 기간은 한 번만. 판정 후에는 그 시점부터 새로 셈.

**통과 조건 (사전 등록):** 세 문제 모두에서
- 채점 기간의 잘못된 판정 ≤ 10%
- 맞는 판정(감지율·맞는 채택)이 기존 맞춤 규칙보다 5%p 넘게 낮지 않을 것

## 진행
- **1차:** 오판 허용 α = 5%로 고정하고 m만 보정 → **실패** (아래 1번)
- **2차:** α도 같은 보정 규칙으로 고름. "가장 민감한 α부터, 가장 작은 m부터 → 보정 기간 잘못된 판정 ≤ 10%인 첫 조합" (아래 2번)

데이터는 모두 저장소에서 내려받습니다(키 불필요). 문제별 코드는 이름이 겹치지 않게 모듈 파일로 씁니다.


In [ ]:
%%writefile unified_core.py
import numpy as np, pandas as pd
from math import log, sqrt

# ===== 통일 점진 규칙 =====
# "증거의 언제 멈춰도 유효한 하한이 실무 최소 크기 m을 넘으면 판정한다."
U_ALPHA = 0.05      # 판정 단위(한 번의 결정 문제)당 전체 오판 허용, 후보 수(방향 수)로 나눠 씀
FA_TARGET = 0.10    # m을 고르는 규칙: 보정 기간 잘못된 판정 ≤ 10%인 가장 작은 m

def halfwidth(V, V0, alpha):
    """정규 혼합 신뢰 시퀀스(Robbins)의 반폭. V = 지금까지 쌓인 증거의 분산, V0 = 가장 좁게 맞출 분산 규모.
    모든 시점에서 동시에 P(누적 증거 − 참값 > 반폭) ≤ alpha."""
    return sqrt((V + V0) * (log(1 + V / V0) + 2 * log(1 / alpha)))

def judge(S, V, scale, V0, m, alpha):
    """S: 누적 증거 합, V: 그 분산, scale: 크기로 바꾸는 분모. +1(증가 쪽 판정) / −1(감소 쪽 판정) / 0(보류)."""
    hw = halfwidth(V, V0, alpha)
    if (S - hw) / scale > m: return 1
    if (S + hw) / scale < -m: return -1
    return 0

def lower(S, V, scale, V0, alpha):
    return (S - halfwidth(V, V0, alpha)) / scale

M_GRID_U = [0, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.125, 0.15]

def _unified_alarm_v1(e, var, scale, n, m, v0_months=6):
    """경보에 통일 규칙 적용. e: 월별 (실제 − 예측), var: 월별 분산, scale: 월별 예측 규모(이탈률의 분모).
    방향 2개(증가/감소)이므로 오판 허용을 2로 나눔. 첫 판정 월과 방향."""
    S = V = Sc = 0.0
    V0 = float(sum(var[:v0_months]))
    for i in range(n):
        S += e[i]; V += var[i]; Sc += scale[i]
        d = judge(S, V, Sc, V0, m, U_ALPHA / 2)
        if d: return i + 1, d
    return None, 0

def _evaluate_unified_v1(units, years, m, dev, clear, min_size):
    """units: {(지역, 연도): dict(e, var, scale, n, size, dev)} → 감지율·잘못된 경보율 (기존 실험과 같은 이탈/정상 정의)."""
    rows = []
    for (c, y), u in units.items():
        if y not in years or u["size"] < min_size or u["n"] < 12: continue
        mo, s = _unified_alarm_v1(u["e"], u["var"], u["scale"], u["n"], m)
        rows.append(dict(code=c, year=y, dev=u["dev"], alarm_m=mo, alarm_dir=s, size=u["size"]))
    R = pd.DataFrame(rows)
    pos, neg = R[R.dev.abs() >= dev], R[R.dev.abs() < clear]
    hit = pos[pos.alarm_dir == np.sign(pos.dev)]
    return R, dict(이탈해=len(pos), 정상해=len(neg), 감지율=round(len(hit) / max(len(pos), 1), 3),
                   상반기내_감지율=round(float((hit.alarm_m <= 6).sum()) / max(len(pos), 1), 3),
                   감지월_중앙값=float(hit.alarm_m.median()) if len(hit) else np.nan,
                   잘못된경보율=round(float((neg.alarm_dir != 0).mean()), 3))

def pick_m(units, cal, dev, clear, min_size):
    for m in M_GRID_U:
        _, s = _evaluate_unified_v1(units, cal, m, dev, clear, min_size)
        if s["잘못된경보율"] <= FA_TARGET: return m, s
    return m, s

# ===== 통일 규칙 2차: 민감도(α)와 실무 크기(m)를 같은 보정 규칙으로 =====
# 보정 기간에서 '가장 민감한 α부터, 가장 작은 m부터' 차례로 시험해 잘못된 판정 ≤ 10%가 되는 첫 조합을 쓴다.
A_GRID = [0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.01]

def unified_alarm(e, var, scale, n, m, alpha=None, v0_months=6):
    alpha = U_ALPHA if alpha is None else alpha
    S = V = Sc = 0.0
    V0 = float(sum(var[:v0_months]))
    for i in range(n):
        S += e[i]; V += var[i]; Sc += scale[i]
        d = judge(S, V, Sc, V0, m, alpha / 2)
        if d: return i + 1, d
    return None, 0

def evaluate_unified(units, years, m, dev, clear, min_size, alpha=None):
    rows = []
    for (c, y), u in units.items():
        if y not in years or u["size"] < min_size or u["n"] < 12: continue
        mo, s = unified_alarm(u["e"], u["var"], u["scale"], u["n"], m, alpha)
        rows.append(dict(code=c, year=y, dev=u["dev"], alarm_m=mo, alarm_dir=s, size=u["size"]))
    R = pd.DataFrame(rows)
    pos, neg = R[R.dev.abs() >= dev], R[R.dev.abs() < clear]
    hit = pos[pos.alarm_dir == np.sign(pos.dev)]
    return R, dict(이탈해=len(pos), 정상해=len(neg), 감지율=round(len(hit) / max(len(pos), 1), 3),
                   상반기내_감지율=round(float((hit.alarm_m <= 6).sum()) / max(len(pos), 1), 3),
                   감지월_중앙값=float(hit.alarm_m.median()) if len(hit) else np.nan,
                   잘못된경보율=round(float((neg.alarm_dir != 0).mean()), 3))

def pick_alpha_m(units, cal, dev, clear, min_size):
    for m in M_GRID_U:
        for a in A_GRID:
            _, s = evaluate_unified(units, cal, m, dev, clear, min_size, a)
            if s["잘못된경보율"] <= FA_TARGET: return a, m, s
    return a, m, s


In [ ]:
%%writefile epl_rules.py
from unified_core import *
import json, os, urllib.request
from math import exp, factorial
import numpy as np, pandas as pd

SEASONS = [f"{y}-{str(y + 1)[2:]}" for y in range(2010, 2027)]
EPL_URLS = ["https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/data_epl/en1_{s}.json",
            "https://raw.githubusercontent.com/openfootball/football.json/master/{s}/en.1.json"]

def _norm(t):
    t = t.strip()
    for suf in (" AFC", " FC"):
        if t.endswith(suf): t = t[:-len(suf)]
    return t[4:] if t.startswith("AFC ") else t

def load_epl(local="data_epl"):
    rows = []
    for s in SEASONS:
        f = os.path.join(local, f"en1_{s}.json")
        if os.path.exists(f):
            ms = json.load(open(f, encoding="utf-8"))["matches"]
        else:
            for u in EPL_URLS:
                try: ms = json.loads(urllib.request.urlopen(u.format(s=s), timeout=30).read())["matches"]; break
                except Exception: ms = None
        for x in ms:
            sc = x.get("score"); ft = sc.get("ft") if isinstance(sc, dict) else (sc if isinstance(sc, list) and len(sc) == 2 else None)
            if not ft: continue
            date = x["date"]
            if s == "2019-20" and date < "2019-08-01": date = "2020" + date[4:]   # 원본 날짜 오류 수정
            rows.append(dict(season=s, date=date, home=_norm(x["team1"]), away=_norm(x["team2"]), hg=ft[0], ag=ft[1]))
    df = pd.DataFrame(rows).sort_values("date", kind="stable").reset_index(drop=True)
    df["y"] = np.select([df.hg > df.ag, df.hg == df.ag], [0, 1], 2)
    return df

# llm-reliability/llmrel/epl.py 의 시작 규칙과 후보 격자 (xG 관련 g 제외)
START = dict(k=0.05, h=0.25, c=0.8, p=-0.2, rho=0.0, w=0.0, cap=99.0, e=1.0)
GRID = {"k": [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.1], "h": [0.1, 0.15, 0.2, 0.25, 0.3, 0.35],
        "c": [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], "p": [0.0, -0.1, -0.2, -0.3, -0.4], "rho": [0.0, 0.05, 0.1, 0.15, 0.2, 0.25],
        "w": [0.0, 0.05, 0.1, 0.15, 0.2, 0.3], "cap": [99.0, 4.0, 3.0, 2.5, 2.0], "e": [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]}
BASE = np.array([0.46, 0.25, 0.29])
MAXG = 10
_G = np.arange(MAXG + 1); _F = np.array([factorial(i) for i in range(MAXG + 1)], float)

def run_model(df, k, h, c, p, rho, w, cap, e, early=6, mu0=0.3):
    """epl.run과 같은 온라인 포아송 (골 모드). 경기별 RPS(결과)와 스코어 로그손실(정확한 스코어)을 함께 반환."""
    a, d, played_n = {}, {}, {}
    mu = mu0; season = None
    rps = np.zeros(len(df)); nll = np.zeros(len(df))
    teams_by = {s: set(g.home) | set(g.away) for s, g in df.groupby("season")}
    for i, r in enumerate(df.itertuples()):
        if r.season != season:
            for t in list(a): a[t] *= c; d[t] *= c
            for t in teams_by[r.season] - set(a): a[t], d[t] = p, p
            season = r.season; played_n = {}
        lh = exp(mu + h + a[r.home] - d[r.away]); la = exp(mu + a[r.away] - d[r.home])
        ph = np.exp(-lh) * lh ** _G / _F; pa = np.exp(-la) * la ** _G / _F
        m = np.outer(ph, pa); m[np.diag_indices_from(m)] *= (1 + rho); m /= m.sum()
        P = (1 - w) * np.array([np.tril(m, -1).sum(), np.trace(m), np.triu(m, 1).sum()]) + w * BASE
        o = np.zeros(3); o[r.y] = 1
        rps[i] = ((np.cumsum(P)[:2] - np.cumsum(o)[:2]) ** 2).sum() / 2
        nll[i] = -np.log(max(m[min(r.hg, MAXG), min(r.ag, MAXG)], 1e-12))     # 스코어 확률 (w 보정 전 분포)
        th, ta = min(r.hg, cap), min(r.ag, cap)
        eh, ea = th - lh, ta - la
        kh = k * (e if played_n.get(r.home, 0) < early else 1.0); ka = k * (e if played_n.get(r.away, 0) < early else 1.0)
        a[r.home] += kh * eh; d[r.away] -= ka * eh; a[r.away] += ka * ea; d[r.home] -= kh * ea
        played_n[r.home] = played_n.get(r.home, 0) + 1; played_n[r.away] = played_n.get(r.away, 0) + 1
        mu += k * 0.05 * (eh + ea)
    return rps, nll

from math import log, sqrt

ALPHA = 0.05      # 전체 오채택 허용 (후보 수로 나눠 씀)
SEASON = 380      # 기존 방식의 판정 단위: 한 시즌
CHECK = 10        # 빠른 방식의 판정 주기: 한 라운드(10경기)
MIN_N = 30        # 분산 추정을 위한 최소 경기 수
N0 = 200          # 신뢰 시퀀스가 가장 좁아지게 맞추는 경기 수

def cs_lower(m, s, n, alpha, n0=N0):
    """언제 멈춰도 유효한 신뢰 시퀀스(정규 혼합)의 하한. m=평균, s=표준편차, n=표본 수."""
    rho2 = (-2 * log(alpha) + log(-2 * log(alpha) + 1)) / n0
    return m - s * sqrt(2 * (n * rho2 + 1) / (n * n * rho2) * log(sqrt(n * rho2 + 1) / alpha))

def fixed_lower(m, s, n):
    """고정 표본 95% 하한 (tuning.py의 부트스트랩 95% 구간을 정규 근사)."""
    return m - 1.96 * s / sqrt(n)

def decide(D, P, n, method, K):
    """경기 n개가 쌓인 시점의 판정. D, P: 후보별 (합, 제곱합) — D는 결과(RPS) 차이, P는 스코어 로그손실 차이.
    차이 = 현재 규칙 손실 − 후보 손실 (양수면 후보가 좋음). 채택할 후보 번호 또는 None."""
    def stats(S):
        out = []
        for s1, s2 in S:
            m = s1 / n; v = max(s2 / n - m * m, 1e-18) * n / (n - 1)
            out.append((m, sqrt(v)))
        return out
    if method in ("흔한 방식", "기존 점진법"):
        if n % SEASON: return None
        st = stats(D)
        if method == "흔한 방식":
            j = max(range(K), key=lambda i: st[i][0]); return j if st[j][0] > 0 else None
        lows = [fixed_lower(m, s, n) for m, s in st]
        j = max(range(K), key=lambda i: lows[i]); return j if lows[j] > 0 else None
    if n < MIN_N or n % CHECK: return None
    a = ALPHA / K
    if method == "빠른-순차":
        st = stats(D)
        lows = [cs_lower(m, s, n, a) for m, s in st]
        j = max(range(K), key=lambda i: lows[i]); return j if lows[j] > 0 else None
    if method in ("빠른-순차+스코어", "빠른-순차+스코어+안전장치"):
        sp, sd = stats(P), stats(D)
        lows = [cs_lower(m, s, n, a) if s > 1e-12 else -1 for m, s in sp]
        j = max(range(K), key=lambda i: lows[i])
        if lows[j] <= 0: return None
        if method.endswith("안전장치") and sd[j][0] < 0: return None   # 스코어로 빨리 판정하되, 결과(RPS)가 나빠지는 중이면 보류
        return j
    raise ValueError(method)

METHODS = ["흔한 방식", "기존 점진법", "빠른-순차", "빠른-순차+스코어", "빠른-순차+스코어+안전장치"]

import numpy as np, pandas as pd

def simulate(stats, method, T=1520, reps=500, seed=0, good_min=0.5):
    """정답을 아는 경주. stats: 후보별 실제 효과(×1000)와 경기별 잡음(EPL에서 잰 값).
    첫 채택이 진짜 좋은 후보(효과 ≥ good_min)인지, 해로운 후보(효과 < 0)인지, 몇 경기 만인지."""
    rng = np.random.default_rng(seed)
    mu_r, mu_n = stats.rps_tru.to_numpy() / 1000, stats.nll_eff.to_numpy() / 1000
    sd_r, sd_n, rho = stats.sd_rps.to_numpy() / 1000, stats.sd_nll.to_numpy() / 1000, stats["corr"].fillna(0).to_numpy()
    K = len(stats); good = stats.rps_tru.to_numpy() >= good_min; bad = stats.rps_tru.to_numpy() < 0
    res = []
    for _ in range(reps):
        z1 = rng.standard_normal((T, K)); z2 = rho * z1 + np.sqrt(1 - rho ** 2) * rng.standard_normal((T, K))
        dr = mu_r + sd_r * z1; dn = mu_n + sd_n * z2
        cr, cr2, cn, cn2 = (np.cumsum(x, 0) for x in (dr, dr ** 2, dn, dn ** 2))
        pick, when = None, None
        for n in range(1, T + 1):
            D = list(zip(cr[n - 1], cr2[n - 1])); P = list(zip(cn[n - 1], cn2[n - 1]))
            j = decide(D, P, n, method, K)
            if j is not None: pick, when = j, n; break
        res.append(dict(pick=pick, when=when, good=pick is not None and good[pick], bad=pick is not None and bad[pick]))
    R = pd.DataFrame(res)
    g = R[R.good]
    return dict(방식=method, 맞는_채택=round(R.good.mean(), 3), 해로운_채택=round(R.bad.mean(), 3),
                채택_없음=round(R.pick.isna().mean(), 3), 맞는_채택까지_경기_중앙값=float(g.when.median()) if len(g) else np.nan,
                맞는_채택까지_시즌=round(float(g.when.median()) / 380, 2) if len(g) else np.nan)

df = load_epl()
DECIDE = ("2018-08-01", "2022-08-01")   # 실시간처럼 흘려보내며 규칙을 바꾸는 기간 (4시즌)
TRUTH = ("2022-08-01", "2099-01-01")    # 채점에만 쓰는 기간 (2022-23 ~ 2026-27 시즌 초)

_cache = {}
def losses(params):
    key = tuple(sorted(params.items()))
    if key not in _cache:
        _cache[key] = run_model(df, **params)
    return _cache[key]

def neighbors(cur):
    out = []
    for p, vals in GRID.items():
        i = vals.index(cur[p])
        for j in (i - 1, i + 1):
            if 0 <= j < len(vals): out.append((f"{p}={vals[j]}", {**cur, p: vals[j]}))
    return out

def candidate_stats():
    """시작 규칙의 이웃 후보별: 결정 기간·진실 기간의 효과(×1000)와 경기별 잡음. 시뮬레이션의 '정답'으로 씀."""
    dec = ((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy(); tru = ((df.date >= TRUTH[0]) & (df.date < TRUTH[1])).to_numpy()
    r0, n0 = losses(START)
    rows = []
    for name, c in neighbors(START):
        r, n = losses(c); dr, dn = r0 - r, n0 - n
        rows.append(dict(후보=name, rps_dec=dr[dec].mean() * 1000, rps_tru=dr[tru].mean() * 1000,
                         rps_tru_t=dr[tru].mean() / dr[tru].std() * np.sqrt(tru.sum()),
                         nll_dec=dn[dec].mean() * 1000, nll_eff=dn[tru].mean() * 1000,
                         sd_rps=dr[dec].std() * 1000, sd_nll=dn[dec].std() * 1000,
                         corr=np.corrcoef(dr[dec], dn[dec])[0, 1] if dn[dec].std() > 0 else 0.0))
    return pd.DataFrame(rows)

def replay(method):
    """결정 기간(2018-19 ~ 2021-22)을 경기 순서대로 흘려보내며 규칙을 바꾼다. 채택하면 그 뒤 경기부터 새로 센다(실시간과 같은 조건)."""
    idx = np.flatnonzero(((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy())
    cur, start, log_ = dict(START), 0, []
    while start < len(idx):
        cands = neighbors(cur); K = len(cands)
        r0, n0 = losses(cur)
        seg = idx[start:]
        dr = np.array([r0[seg] - losses(c)[0][seg] for _, c in cands]).T
        dn = np.array([n0[seg] - losses(c)[1][seg] for _, c in cands]).T
        cr, cr2, cn, cn2 = (np.cumsum(x, 0) for x in (dr, dr ** 2, dn, dn ** 2))
        pick = None
        for n in range(1, len(seg) + 1):
            # 기존 방식의 판정 시점은 '시즌 끝'(누적 경기 수가 아니라 실제 시즌 경계)에 맞춤
            if method in ("흔한 방식", "기존 점진법"):
                last = seg[n - 1]
                if n < len(seg) and df.season.iat[seg[n]] == df.season.iat[last]: continue
                st = [(cr[n - 1, i] / n, np.sqrt(max(cr2[n - 1, i] / n - (cr[n - 1, i] / n) ** 2, 1e-18) * n / (n - 1))) for i in range(K)]
                if method == "흔한 방식":
                    j = max(range(K), key=lambda i: st[i][0]); j = j if st[j][0] > 0 else None
                else:
                    lows = [fixed_lower(m, s, n) for m, s in st]; j = max(range(K), key=lambda i: lows[i]); j = j if lows[j] > 0 else None
            else:
                j = decide(list(zip(cr[n - 1], cr2[n - 1])), list(zip(cn[n - 1], cn2[n - 1])), n, method, K)
            if j is not None:
                pick = (n, j); break
        if pick is None: break
        n, j = pick
        log_.append(dict(방식=method, 날짜=df.date.iat[seg[n - 1]], 시즌=df.season.iat[seg[n - 1]], 판정까지_경기=n, 바꾼규칙=cands[j][0]))
        cur, start = cands[j][1], start + n
    return cur, pd.DataFrame(log_)

def offline_hill_climb():
    """기존 점진법 원형(tuning.py): 결정 기간 4시즌을 다 모은 뒤 한 번에 '가장 확실한 개선'을 반복 채택 (부트스트랩 95%)."""
    msk = ((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy()
    rng = np.random.default_rng(0)
    cur = dict(START); log_ = []
    while True:
        r0 = losses(cur)[0][msk]; best = None
        for name, c in neighbors(cur):
            d = r0 - losses(c)[0][msk]
            boots = d[rng.integers(0, len(d), (1000, len(d)))].mean(1)
            lo = np.percentile(boots, 2.5)
            if best is None or lo > best[0]: best = (lo, name, c)
        if best[0] <= 0: break
        log_.append(dict(방식="기존 점진법 (4시즌 모아 한 번에)", 날짜=DECIDE[1], 시즌="2021-22 끝", 판정까지_경기=int(msk.sum()), 바꾼규칙=best[1]))
        cur = best[2]
    return cur, pd.DataFrame(log_)

def truth_gain(params, block=10, reps=2000, seed=0):
    """진실 기간 RPS 개선(시작 규칙 대비, ×1000)과 경기 10개 블록 부트스트랩 95% 구간."""
    tru = ((df.date >= TRUTH[0]) & (df.date < TRUTH[1])).to_numpy()
    d = (losses(START)[0] - losses(params)[0])[tru] * 1000
    nb = len(d) // block; b = d[: nb * block].reshape(nb, block).mean(1)
    m = np.random.default_rng(seed).choice(b, (reps, nb)).mean(1)
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

def realized_during(log_):
    """결정 기간 동안 '그때그때 쓰고 있던 규칙'의 RPS 개선(시작 규칙 대비, ×1000). 일찍 채택할수록 일찍 이득."""
    msk = ((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy()
    idx = np.flatnonzero(msk); cur = dict(START); base = losses(START)[0]
    changes = {} if log_.empty else {d: r for d, r in zip(log_.날짜, log_.바꾼규칙)}
    out = np.zeros(len(idx))
    for t, i in enumerate(idx):
        out[t] = (base[i] - losses(cur)[0][i]) * 1000
        dte = df.date.iat[i]
        if dte in changes and (t + 1 == len(idx) or df.date.iat[idx[t + 1]] != dte):
            p, v = changes[dte].split("="); cur = {**cur, p: type(START[p])(float(v))}
    return out.mean()

WINDOWS = {   # (결정 기간, 채점 기간) — 결과를 보기 전에 고정
    "W1 결정 11-15 → 채점 15-19": (("2011-08-01", "2015-08-01"), ("2015-08-01", "2019-08-01")),
    "W2 결정 15-19 → 채점 19-23": (("2015-08-01", "2019-08-01"), ("2019-08-01", "2023-08-01")),
    "W3 결정 18-22 → 채점 22-27": (("2018-08-01", "2022-08-01"), ("2022-08-01", "2099-01-01")),
}
GOOD_T = 2.0   # 채점 기간 t ≥ 2 → 좋은 후보, 효과 < 0 → 해로운 후보, 나머지는 중립

def simulate_window(stats, method, reps=500, seed=0):
    """simulate와 같지만 좋은/해로운 후보를 채점 기간 t 기준으로 정함."""
    s = stats.copy()
    s["rps_tru"] = np.where(s.rps_tru_t >= GOOD_T, np.maximum(s.rps_tru, 0.5), s.rps_tru)   # 표시용 아님: good_min 판정을 위해 좋은 후보 효과를 0.5 이상으로 맞춤
    r = simulate(stats, method, reps=reps, seed=seed, good_min=np.inf)
    # good 정의를 t 기준으로 다시 계산
    rng = np.random.default_rng(seed)
    mu_r, mu_n = stats.rps_tru.to_numpy() / 1000, stats.nll_eff.to_numpy() / 1000
    sd_r, sd_n, rho = stats.sd_rps.to_numpy() / 1000, stats.sd_nll.to_numpy() / 1000, stats["corr"].fillna(0).to_numpy()
    K = len(stats); T = 1520
    good = (stats.rps_tru_t >= GOOD_T).to_numpy(); bad = (stats.rps_tru < 0).to_numpy()
    res = []
    for _ in range(reps):
        z1 = rng.standard_normal((T, K)); z2 = rho * z1 + np.sqrt(1 - rho ** 2) * rng.standard_normal((T, K))
        dr = mu_r + sd_r * z1; dn = mu_n + sd_n * z2
        cr, cr2, cn, cn2 = (np.cumsum(x, 0) for x in (dr, dr ** 2, dn, dn ** 2))
        pick = when = None
        for n in range(1, T + 1):
            j = decide(list(zip(cr[n - 1], cr2[n - 1])), list(zip(cn[n - 1], cn2[n - 1])), n, method, K)
            if j is not None: pick, when = j, n; break
        res.append(dict(pick=pick, when=when, good=pick is not None and good[pick], bad=pick is not None and bad[pick]))
    R = pd.DataFrame(res); g = R[R.good]
    return dict(방식=method, 좋은후보수=int(good.sum()), 맞는_채택=round(R.good.mean(), 3), 해로운_채택=round(R.bad.mean(), 3),
                채택_없음=round(R.pick.isna().mean(), 3), 맞는_채택까지_시즌=round(float(g.when.median()) / 380, 2) if len(g) else np.nan)

def replicate():
    global DECIDE, TRUTH
    sims, reps_, cands = [], [], []
    for wname, (dec, tru) in WINDOWS.items():
        DECIDE, TRUTH = dec, tru
        S = candidate_stats(); S.insert(0, "기간", wname); cands.append(S)
        good = set(S.후보[S.rps_tru_t >= GOOD_T]); bad = set(S.후보[S.rps_tru < 0])
        for m in ["기존 점진법", "빠른-순차+스코어", "빠른-순차+스코어+안전장치"]:
            r = simulate_window(S, m); r["기간"] = wname; sims.append(r)
        for m in ["흔한 방식", "기존 점진법", "빠른-순차+스코어", "빠른-순차+스코어+안전장치"]:
            cur, L = replay(m)
            first_good = L[L.바꾼규칙.isin(good)].판정까지_경기.cumsum() if len(L) else pd.Series(dtype=float)
            # 첫 '좋은 후보' 채택까지 경기 수 (앞선 채택 포함 누적)
            n_cum = L.판정까지_경기.cumsum().to_numpy() if len(L) else []
            fg = next((int(n_cum[i]) for i, ch in enumerate(L.바꾼규칙) if ch in good), None) if len(L) else None
            g_, lo, hi = truth_gain(cur)
            reps_.append(dict(기간=wname, 방식=m, 채택=" → ".join(L.바꾼규칙) if len(L) else "(없음)",
                              해로운_채택=sum(ch in bad for ch in L.바꾼규칙) if len(L) else 0,
                              첫_좋은후보_채택까지_경기=fg, 채점기간_개선=f"{g_:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
        cur, L = offline_hill_climb(); g_, lo, hi = truth_gain(cur)
        reps_.append(dict(기간=wname, 방식="기존 점진법 (4시즌 모아 한 번에)", 채택=" → ".join(L.바꾼규칙) if len(L) else "(없음)",
                          해로운_채택=sum(ch in bad for ch in L.바꾼규칙) if len(L) else 0,
                          첫_좋은후보_채택까지_경기=1520 if any(ch in good for ch in L.바꾼규칙) else None,
                          채점기간_개선=f"{g_:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
    return pd.concat(cands), pd.DataFrame(sims), pd.DataFrame(reps_)

U_M_EPL, U_A_EPL = 0.0, U_ALPHA   # pick_epl()로 보정 기간(W1)에서 정함

def decide_unified(D, P, n, K, check=CHECK, n0=N0):
    """EPL 규칙 개선에 통일 규칙 적용: 신호 = 스코어 로그손실 차이, 안전장치 = 결과(RPS) 평균이 음수면 보류."""
    if n < MIN_N or n % check: return None
    a = U_A_EPL / K
    best, bl = None, -1e9
    for i, ((s1, s2), (r1, _)) in enumerate(zip(P, D)):
        mean = s1 / n; var = max(s2 / n - mean * mean, 1e-18) * n / (n - 1)
        if var < 1e-15: continue
        lb = lower(s1, n * var, n, n0 * var, a)
        if lb > bl: best, bl = i, lb
    if best is None or bl <= U_M_EPL: return None
    if D[best][0] / n < 0: return None
    return best

_old_decide = decide
def decide(D, P, n, method, K):
    if method == "통일 규칙": return decide_unified(D, P, n, K)
    return _old_decide(D, P, n, method, K)

def pick_epl(reps=300):
    """보정 기간 W1에서: 가장 민감한 α부터, 가장 작은 m부터 → '좋은 후보가 아닌 것을 채택' ≤ 10%인 첫 조합."""
    global U_A_EPL, U_M_EPL, DECIDE, TRUTH
    DECIDE, TRUTH = WINDOWS["W1 결정 11-15 → 채점 15-19"]
    S = candidate_stats()
    for m in [0, 0.0005, 0.001, 0.002]:
        for a in A_GRID:
            U_A_EPL, U_M_EPL = a, m
            r = simulate_window(S, "통일 규칙", reps=reps)
            wrong = 1 - r["맞는_채택"] - r["채택_없음"]
            if wrong <= FA_TARGET: return a, m, r, wrong
    return a, m, r, wrong


In [ ]:
%%writefile migration_alarm.py
import os, numpy as np, pandas as pd

DATA = "data_kosis/youth_migration_monthly_sigungu_2005_2026.csv"
DATA_URL = ("https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/"
            "experiments/data_kosis/youth_migration_monthly_sigungu_2005_2026.csv")

# ---- 미리 정한 것 (채점 기간 결과를 보기 전에 고정) ----
CAL = range(2010, 2018)      # 기준을 정하는 기간 (예측 연도)
TEST = range(2018, 2026)     # 채점 기간
LIVE = 2026                  # 실시간 (1~8월 공표분)
MIN_SIZE = 2000              # 직전 연도 청년(20~39세) 전입 2,000명 이상 시군구만 평가
DEV_Q, CLEAR_Q = 0.80, 0.50  # 이탈 = 보정 기간 |이탈률| 상위 20%, 정상 = 하위 50% (기준값은 보정 기간에서만 계산)
K = 0.25                     # CUSUM 허용 폭
MAX_FA = 0.10                # 보정 기간 정상 해의 잘못된 경보율 10% 이하가 되는 가장 민감한 기준
LAG = 1                      # 월별 인구이동 공표 지연 (다음 달 말 공표 → 약 1개월)

def load(path=DATA):
    d = pd.read_csv(path if os.path.exists(path) else DATA_URL, dtype={"code": str, "ym": str})
    d["year"] = d.ym.str[:4].astype(int); d["month"] = d.ym.str[4:].astype(int)
    return d

def tables(d):
    """지역 × (연, 월) 전입·전출 표 (20~39세 합계)."""
    wide = lambda col: d.pivot_table(index="code", columns=["year", "month"], values=col)
    return wide("in_"), wide("out")

def forecasts(IN, OUT, years_needed=6):
    """플랜 A: 전입·전출 각각 '작년 연간 × 최근 5년 평균 월별 비중'. 순이동 예측 = 전입 예측 − 전출 예측.
    {(지역, 연도): dict(mu_in, mu_out, a_in, a_out, prev_in, n_months)}"""
    out = {}
    years = sorted({y for y, _ in IN.columns})
    def plan_a(row, Y):
        M = np.array([[row.get((y, m), np.nan) for m in range(1, 13)] for y in [Y - i for i in range(1, 6)]])
        if np.isnan(M).any(): return None, None
        shares = (M / M.sum(1, keepdims=True)).mean(0)
        return M[0].sum() * shares, M[0].sum()
    for code in IN.index:
        if code not in OUT.index: continue
        ri, ro = IN.loc[code], OUT.loc[code]
        for Y in years:
            if any(Y - i not in years for i in range(1, 6)): continue
            mi, prev_in = plan_a(ri, Y); mo, _ = plan_a(ro, Y)
            if mi is None or mo is None: continue
            ai = np.array([ri.get((Y, m), np.nan) for m in range(1, 13)])
            ao = np.array([ro.get((Y, m), np.nan) for m in range(1, 13)])
            n = int(np.sum(~np.isnan(ai) & ~np.isnan(ao)))
            if n == 0: continue
            out[(code, Y)] = dict(mu_in=mi, mu_out=mo, a_in=ai, a_out=ao, prev_in=prev_in, n=n)
    return out

def dev_rate(f, upto=12):
    """이탈률 = (실제 순이동 − 예측 순이동) 누적 / 예측 전입 누적."""
    s = slice(0, upto)
    return (np.nansum(f["a_in"][s] - f["a_out"][s]) - np.sum(f["mu_in"][s] - f["mu_out"][s])) / np.sum(f["mu_in"][s])

def phi_from(F, years):
    r = []
    for (c, y), f in F.items():
        if y in years and f["prev_in"] >= MIN_SIZE and f["n"] == 12:
            v = f["mu_in"] + f["mu_out"]
            r.append((((f["a_in"] - f["a_out"]) - (f["mu_in"] - f["mu_out"])) ** 2 / v).mean())
    return float(np.mean(r))

def thresholds(F):
    x = np.abs([dev_rate(f) for (c, y), f in F.items() if y in CAL and f["prev_in"] >= MIN_SIZE and f["n"] == 12])
    return float(np.quantile(x, DEV_Q)), float(np.quantile(x, CLEAR_Q))

def alarm(f, phi, h, m_min=0.0, k=K):
    """순이동 잔차의 양방향 CUSUM + (선택) 누적 이탈률 ≥ m_min. 첫 경보 월과 방향(+1 순유입 증가 / −1 순유출 증가)."""
    z = ((f["a_in"] - f["a_out"]) - (f["mu_in"] - f["mu_out"])) / np.sqrt(phi * (f["mu_in"] + f["mu_out"]))
    cp = cm = 0.0
    for m in range(f["n"]):
        cp = max(0.0, cp + z[m] - k); cm = max(0.0, cm - z[m] - k)
        r = dev_rate(f, m + 1)
        if cp > h and r >= m_min: return m + 1, 1
        if cm > h and r <= -m_min: return m + 1, -1
    return None, 0

def evaluate(F, years, phi, h, dev, clear, m_min=0.0):
    rows = []
    for (c, y), f in F.items():
        if y not in years or f["prev_in"] < MIN_SIZE or f["n"] < 12: continue
        m, s = alarm(f, phi, h, m_min)
        rows.append(dict(code=c, year=y, dev=dev_rate(f), alarm_m=m, alarm_dir=s, prev_in=f["prev_in"]))
    R = pd.DataFrame(rows)
    pos, neg = R[R.dev.abs() >= dev], R[R.dev.abs() < clear]
    hit = pos[pos.alarm_dir == np.sign(pos.dev)]
    return R, dict(지역연도=len(R), 이탈해=len(pos), 정상해=len(neg),
                   감지율=round(len(hit) / max(len(pos), 1), 3),
                   상반기내_감지율=round(float((hit.alarm_m <= 6).sum()) / max(len(pos), 1), 3),
                   감지월_중앙값=float(hit.alarm_m.median()) if len(hit) else np.nan,
                   잘못된경보율=round(float((neg.alarm_dir != 0).mean()), 3) if len(neg) else np.nan)

def by_size(R, dev, clear, cuts=(0, 5000, 15000, 1e9), labels=("2천~5천", "5천~1.5만", "1.5만 이상")):
    R = R.copy()
    R["규모"] = pd.cut(R.prev_in, list(cuts), labels=list(labels))
    R["이탈"] = R.dev.abs() >= dev; R["정상"] = R.dev.abs() < clear
    R["감지"] = R.이탈 & (R.alarm_dir == np.sign(R.dev))
    def s(x):
        return pd.Series(dict(지역연도=len(x), 이탈해=int(x.이탈.sum()), 감지율=round(x.감지.sum() / max(x.이탈.sum(), 1), 2),
                              감지월_중앙값=x[x.감지].alarm_m.median(), 정상해=int(x.정상.sum()),
                              잘못된경보율=round(float((x[x.정상].alarm_dir != 0).mean()), 2)))
    return pd.concat([R.groupby("규모", observed=True).apply(s), s(R).to_frame("전체").T])

d = load()
names = d.drop_duplicates("code").set_index("code").name
IN, OUT = tables(d)
F = forecasts(IN, OUT)
phi = phi_from(F, CAL)
DEV, CLEAR = thresholds(F)
H_GRID = [2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10, 12]
for h in H_GRID:                                   # 보정 기간: 정상 해 잘못된 경보율 ≤ 10%인 가장 민감한 h
    _, cal = evaluate(F, CAL, phi, h, DEV, CLEAR)
    if cal["잘못된경보율"] <= MAX_FA: break
M_GRID = [0, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1]
for m_cal in M_GRID:                               # 보정 기간: 가장 큰 규모 구간의 잘못된 경보율 ≤ 10%인 가장 작은 실무 크기
    Rc, _ = evaluate(F, CAL, phi, h, DEV, CLEAR, m_cal)
    if by_size(Rc, DEV, CLEAR).iloc[-2]["잘못된경보율"] <= MAX_FA: break
print(f"시군구 {IN.shape[0]}곳 · 과분산 φ {phi:.2f} · 이탈 기준 |이탈률| ≥ {DEV:.1%}, 정상 < {CLEAR:.1%} (보정 기간 분포)")
print(f"보정으로 고른 값: CUSUM h = {h}, 실무 크기 m = {m_cal:.0%}")
R0, t0 = evaluate(F, TEST, phi, h, DEV, CLEAR)
R1, t1 = evaluate(F, TEST, phi, h, DEV, CLEAR, m_cal)
SIDO = {"11": "서울", "12": "전남광주", "26": "부산", "27": "대구", "28": "인천", "29": "광주", "30": "대전", "31": "울산",
        "36": "세종", "41": "경기", "43": "충북", "44": "충남", "46": "전남", "47": "경북", "48": "경남", "50": "제주",
        "51": "강원", "52": "전북"}
label = lambda c: f"{SIDO.get(c[:2], c[:2])} {names[c].replace(' ', '')}"

def live_alarms(F, year=LIVE):
    """올해 공표분까지의 실시간 경보 목록 (보정 기간에서 고른 h·m 그대로).
    상태: 지금도 같은 방향으로 실무 크기 이상 벗어나 있으면 '유지', 아니면 '해소'."""
    rows = []
    for (c, y), f in F.items():
        if y != year or f["prev_in"] < MIN_SIZE: continue
        m, s = alarm(f, phi, h, m_cal)
        if s == 0: continue
        n = f["n"]; r = dev_rate(f, n)
        net = np.nansum(f["a_in"][:n] - f["a_out"][:n]); net_hat = np.sum(f["mu_in"][:n] - f["mu_out"][:n])
        rows.append(dict(지역=label(c), 반영=f"1~{n}월", 경보월=f"{m}월", 방향="순유입↑" if s > 0 else "순유출↑",
                         실제=int(net), 예측=int(round(net_hat)), 차이=int(round(net - net_hat)), 이탈률=f"{r:+.1%}",
                         상태="유지" if np.sign(r) == s and abs(r) >= m_cal else "해소"))
    return pd.DataFrame(rows).sort_values(["상태", "차이"], key=lambda s: s if s.name == "상태" else -s.abs()).reset_index(drop=True)

from unified_core import evaluate_unified, pick_alpha_m, _evaluate_unified_v1, pick_m
UNITS = {k: dict(e=np.nan_to_num((f["a_in"] - f["a_out"]) - (f["mu_in"] - f["mu_out"])), var=phi * (f["mu_in"] + f["mu_out"]),
                 scale=f["mu_in"], n=f["n"], size=f["prev_in"], dev=dev_rate(f) if f["n"] == 12 else np.nan) for k, f in F.items()}
CUSTOM = t1   # 기존 맞춤 규칙(CUSUM + 실무 크기)의 채점 결과
SIZE_CUTS, SIZE_LABELS = [0, 5000, 15000, 1e9], ["2천~5천", "5천~1.5만", "1.5만 이상"]


In [ ]:
%%writefile marriage_alarm.py
import os, numpy as np, pandas as pd

DATA = "data_kosis/marriages_monthly_sigungu_1997_2025.csv"
DATA_URL = ("https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/"
            "experiments/data_kosis/marriages_monthly_sigungu_1997_2025.csv")

# ---- 미리 정한 것 (채점 기간 결과를 보기 전에 고정) ----
CAL = range(2003, 2013)      # 경보 기준을 정하는 기간 (예측 연도)
TEST = range(2013, 2026)     # 채점 기간
MIN_SIZE = 300               # 직전 연도 혼인 300건 이상 시군구만 평가
DEV, CLEAR = 0.10, 0.05      # 연간 실제/예측 −1 이 ±10% 이상 = 이탈 해, ±5% 미만 = 정상 해 (사이는 제외)
K = 0.25                     # CUSUM 허용 폭 (표준화 단위)
MAX_FA = 0.10                # 보정 기간 정상 해의 잘못된 경보율 10% 이하가 되는 가장 민감한 기준을 고름
LAG = 2                      # 시도 월별 인구동향 공표 지연 (개월)
RELEASE_M = 15               # 시군구 월별 확정치 공표 시점: 다음 해 3월쯤 (예측 연도 1월 = 1)

def load(path=DATA):
    d = pd.read_csv(path if os.path.exists(path) else DATA_URL, dtype={"code": str, "ym": str})
    d["year"] = d.ym.str[:4].astype(int); d["month"] = d.ym.str[4:].astype(int)
    return d

def wide(d, codes):
    return d[d.code.isin(codes)].pivot_table(index="code", columns=["year", "month"], values="marriages")

def forecasts(W):
    """플랜 A 예측: 작년 연간 건수 × 최근 5년 평균 월별 비중. {(지역, 연도): (월별 예측, 월별 실제, 작년 연간)}"""
    out = {}
    years = sorted({y for y, _ in W.columns})
    for code, row in W.iterrows():
        for Y in years:
            need = [Y - i for i in range(6)]
            if any(y not in years for y in need): continue
            M = np.array([[row.get((y, m), np.nan) for m in range(1, 13)] for y in need])
            if np.isnan(M).any(): continue
            shares = (M[1:] / M[1:].sum(1, keepdims=True)).mean(0)
            out[(code, Y)] = (M[1].sum() * shares, M[0], M[1].sum())
    return out

def relative_forecasts(W, S):
    """시도 대비 예측: 이번 달 시도 실제 × 작년 이 시군구의 시도 내 비중. 전국·시도 흐름을 걸러낸 '지역 고유' 이탈용."""
    out = {}
    years = sorted({y for y, _ in W.columns})
    for code, row in W.iterrows():
        if code[:2] not in S.index: continue
        srow = S.loc[code[:2]]
        for Y in years:
            if Y - 1 not in years: continue
            g = lambda r, y: np.array([r.get((y, m), np.nan) for m in range(1, 13)])
            a, ap, s, sp = g(row, Y), g(row, Y - 1), g(srow, Y), g(srow, Y - 1)
            if np.isnan(np.r_[a, ap, s, sp]).any(): continue
            out[(code, Y)] = (s * ap.sum() / sp.sum(), a, ap.sum())
    return out

def phi_from(F, years, min_size=MIN_SIZE):
    """과분산: 분산이 평균의 몇 배인지 (보정 기간에서만 추정)."""
    return float(np.mean([((a - mu) ** 2 / mu).mean() for (c, y), (mu, a, A) in F.items() if y in years and A >= min_size]))

def cusum_month(mu, actual, phi, h, k=K):
    """1월부터 양방향 CUSUM. 첫 경보 월(1~12)과 방향(+1/−1). 없으면 (None, 0)."""
    z = (actual - mu) / np.sqrt(phi * mu)
    cp = cm = 0.0
    for m in range(12):
        cp = max(0.0, cp + z[m] - k); cm = max(0.0, cm - z[m] - k)
        if cp > h: return m + 1, 1
        if cm > h: return m + 1, -1
    return None, 0

def ytd_month(mu, actual, x):
    """단순 규칙: 1월부터 누적 실제 / 누적 예측 − 1 이 ±x를 넘는 첫 달."""
    r = np.cumsum(actual) / np.cumsum(mu) - 1
    for m in range(12):
        if r[m] > x: return m + 1, 1
        if r[m] < -x: return m + 1, -1
    return None, 0

def evaluate(F, years, rule, min_size=MIN_SIZE):
    rows = []
    for (c, y), (mu, a, A) in F.items():
        if y not in years or A < min_size: continue
        m, s = rule(mu, a)
        rows.append(dict(code=c, year=y, dev=a.sum() / mu.sum() - 1, alarm_m=m, alarm_dir=s))
    R = pd.DataFrame(rows)
    pos, neg = R[R.dev.abs() >= DEV], R[R.dev.abs() < CLEAR]
    hit = pos[pos.alarm_dir == np.sign(pos.dev)]
    return R, dict(지역연도=len(R), 이탈해=len(pos), 정상해=len(neg),
                   감지율=round(len(hit) / max(len(pos), 1), 3),
                   상반기내_감지율=round(float((hit.alarm_m <= 6).sum()) / max(len(pos), 1), 3),
                   감지월_중앙값=float(hit.alarm_m.median()) if len(hit) else np.nan,
                   잘못된경보율=round(float((neg.alarm_dir != 0).mean()), 3) if len(neg) else np.nan)

def pick(F, make_rule, grid, min_size=MIN_SIZE):
    """보정 기간에서 잘못된 경보율 ≤ MAX_FA 인 가장 민감한 기준."""
    for g in grid:
        _, s = evaluate(F, CAL, make_rule(g), min_size)
        if s["잘못된경보율"] <= MAX_FA:
            return g, s
    return g, s

def sido_lead(F, FS, phi_s, h_s):
    """시군구 이탈 해 중 같은 해 소속 시도의 월간 경보(같은 방향)로 미리 안 비율과, 시군구 확정치보다 앞선 개월."""
    rows = []
    for (c, y), (mu, a, A) in F.items():
        if y not in TEST or A < MIN_SIZE: continue
        r = a.sum() / mu.sum() - 1
        if abs(r) < DEV: continue
        smu, sa, _ = FS.get((c[:2], y), (None, None, None))
        if smu is None: continue
        m, s = cusum_month(smu, sa, phi_s, h_s)
        ok = s == np.sign(r)
        rows.append(dict(code=c, year=y, dev=r, sido_alarm_m=m if ok else np.nan,
                         lead=RELEASE_M - (m + LAG) if ok else np.nan))
    return pd.DataFrame(rows)

d = load()
name = d.drop_duplicates("code").set_index("code").name
sgg = [c for c in d.code.unique() if len(c) == 5 and c.endswith("0") and "변동전" not in name[c]]   # 시·군·자치구 (일반구·개편 전 계열 제외)
sido = [c for c in d.code.unique() if len(c) == 2 and c not in ("00", "90")]
W, S = wide(d, sgg), wide(d, sido)
F, FR, FS = forecasts(W), relative_forecasts(W, S), forecasts(S)
phi, phi_r, phi_s = phi_from(F, CAL), phi_from(FR, CAL), phi_from(FS, CAL, 0)

H_GRID = [2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10, 12]
h_abs, cal_abs = pick(F, lambda h: (lambda mu, a: cusum_month(mu, a, phi, h)), H_GRID)
x_ytd, cal_ytd = pick(F, lambda x: (lambda mu, a: ytd_month(mu, a, x)), [0.05, 0.075, 0.1, 0.125, 0.15, 0.2, 0.25, 0.3, 0.4])
h_rel, cal_rel = pick(FR, lambda h: (lambda mu, a: cusum_month(mu, a, phi_r, h)), H_GRID)
h_s, cal_s = pick(FS, lambda h: (lambda mu, a: cusum_month(mu, a, phi_s, h)), H_GRID, min_size=0)

R_abs, t_abs = evaluate(F, TEST, lambda mu, a: cusum_month(mu, a, phi, h_abs))
_, t_ytd = evaluate(F, TEST, lambda mu, a: ytd_month(mu, a, x_ytd))
R_rel, t_rel = evaluate(FR, TEST, lambda mu, a: cusum_month(mu, a, phi_r, h_rel))
_, t_s = evaluate(FS, TEST, lambda mu, a: cusum_month(mu, a, phi_s, h_s), min_size=0)

print(f"시군구 {len(sgg)}곳 · 시도 {len(sido)}곳 · 과분산 φ: 절대 {phi:.2f}, 시도 대비 {phi_r:.2f}, 시도 {phi_s:.1f}")
summary = pd.DataFrame({
    f"시군구 절대 CUSUM (h={h_abs})": t_abs,
    f"시군구 누적 이탈 규칙 (±{x_ytd:.0%})": t_ytd,
    f"시군구 시도 대비 CUSUM (h={h_rel})": t_rel,
    f"시도 절대 CUSUM (h={h_s})": t_s,
}).T
def cusum_practical(mu, actual, phi, h, m_min, k=K):
    """통계 조건(CUSUM > h)과 실무 크기 조건(1월부터 누적 이탈 ≥ m_min)이 같은 달에 함께 충족되는 첫 달."""
    z = (actual - mu) / np.sqrt(phi * mu)
    r = np.cumsum(actual) / np.cumsum(mu) - 1
    cp = cm = 0.0
    for m in range(12):
        cp = max(0.0, cp + z[m] - k); cm = max(0.0, cm - z[m] - k)
        if cp > h and r[m] >= m_min: return m + 1, 1
        if cm > h and r[m] <= -m_min: return m + 1, -1
    return None, 0

def by_size(F, years, rule):
    R, _ = evaluate(F, years, rule)
    R["작년건수"] = [F[(c, y)][2] for c, y in zip(R.code, R.year)]
    R["규모"] = pd.cut(R.작년건수, [0, 1000, 3000, 1e9], labels=["300~1천", "1천~3천", "3천 이상"])
    R["이탈"] = R.dev.abs() >= DEV; R["정상"] = R.dev.abs() < CLEAR
    R["감지"] = R.이탈 & (R.alarm_dir == np.sign(R.dev))
    def s(x):
        return pd.Series(dict(이탈해=int(x.이탈.sum()), 감지율=round(x.감지.sum() / max(x.이탈.sum(), 1), 2),
                              감지월_중앙값=x[x.감지].alarm_m.median(), 정상해=int(x.정상.sum()),
                              잘못된경보율=round(float((x[x.정상].alarm_dir != 0).mean()), 2)))
    return pd.concat([R.groupby("규모", observed=True).apply(s), s(R).to_frame("전체").T])

# 실무 크기 조건: 보정 기간(2003~2012)만 보고 두 가지를 정함
M_PRINCIPLE = CLEAR       # 원칙: '정상 해' 경계(±5%)를 넘을 때만 경보
M_GRID = [0, 0.025, 0.05, 0.075, 0.10, 0.125, 0.15]
for m_cal in M_GRID:      # 보정 규칙: 3천 건 이상 도시의 잘못된 경보율 ≤ 10%가 되는 가장 작은 값
    t = by_size(F, CAL, lambda mu, a, m=m_cal: cusum_practical(mu, a, phi, h_abs, m))
    if t.loc["3천 이상", "잘못된경보율"] <= MAX_FA:
        break
print(f"원칙으로 정한 값 ±{M_PRINCIPLE:.0%}, 보정 기간으로 고른 값 ±{m_cal:.1%} (보정 기간 3천 건 이상 이탈 해는 {int(t.loc['3천 이상','이탈해'])}건뿐)")

from unified_core import evaluate_unified, pick_alpha_m, _evaluate_unified_v1, pick_m
UNITS = {k: dict(e=a - mu, var=phi * mu, scale=mu, n=12, size=A, dev=a.sum() / mu.sum() - 1) for k, (mu, a, A) in F.items()}
DEV_, CLEAR_ = DEV, CLEAR
_, CUSTOM = evaluate(F, TEST, lambda mu, a: cusum_practical(mu, a, phi, h_abs, m_cal))   # 기존 맞춤 규칙(CUSUM + 누적 ±10%)
SIZE_CUTS, SIZE_LABELS = [0, 1000, 3000, 1e9], ["300~1천", "1천~3천", "3천 이상"]


In [ ]:

import importlib, numpy as np, pandas as pd
import unified_core as U, migration_alarm as MG, marriage_alarm as MR
for mod in (U, MG, MR): importlib.reload(mod)

def size_table(R, P):
    R = R.copy(); R["규모"] = pd.cut(R["size"], P.SIZE_CUTS, labels=P.SIZE_LABELS)
    R["이탈"] = R.dev.abs() >= P.DEV; R["정상"] = R.dev.abs() < P.CLEAR; R["감지"] = R.이탈 & (R.alarm_dir == np.sign(R.dev))
    return R.groupby("규모", observed=True).apply(lambda x: pd.Series(dict(
        감지율=round(x.감지.sum() / max(x.이탈.sum(), 1), 2), 잘못된경보율=round(float((x[x.정상].alarm_dir != 0).mean()), 2))))


## 1. 1차 통일 규칙 (α = 5% 고정): 경보 두 문제

In [ ]:

rows = []
for name, P in [("청년 순이동", MG), ("혼인", MR)]:
    m, cal = U.pick_m(P.UNITS, P.CAL, P.DEV, P.CLEAR, P.MIN_SIZE)
    _, t = U._evaluate_unified_v1(P.UNITS, P.TEST, m, P.DEV, P.CLEAR, P.MIN_SIZE)
    rows.append(dict(문제=name, m=m, 보정기간_잘못된경보=cal["잘못된경보율"], 감지율_통일1차=t["감지율"],
                     감지율_기존=P.CUSTOM["감지율"], 잘못된경보_통일1차=t["잘못된경보율"], 잘못된경보_기존=P.CUSTOM["잘못된경보율"]))
pd.DataFrame(rows).set_index("문제")



**1차는 실패했습니다.** 감지율이 기존의 절반 수준이었습니다(청년 순이동 44% vs 85%, 혼인 31% vs 80%).
보정 기간의 잘못된 경보가 이미 0%라서 목표(10%)보다 훨씬 보수적이었습니다.
기존 맞춤 규칙은 민감도(CUSUM 기준)를 보정 기간에서 골랐는데, 1차는 민감도를 고정했기 때문입니다.
→ 2차에서 **민감도도 같은 보정 규칙으로** 고릅니다. 1차 실패를 보고 바꾼 것이지만, 값은 보정 기간만으로 정합니다.


## 2. 2차 통일 규칙: 경보 두 문제

In [ ]:

rows, sizes = [], {}
for name, P in [("청년 순이동", MG), ("혼인", MR)]:
    a, m, cal = U.pick_alpha_m(P.UNITS, P.CAL, P.DEV, P.CLEAR, P.MIN_SIZE)
    R, t = U.evaluate_unified(P.UNITS, P.TEST, m, P.DEV, P.CLEAR, P.MIN_SIZE, a)
    rows.append(dict(문제=name, α=a, m=m, 감지율_통일=t["감지율"], 감지율_기존=P.CUSTOM["감지율"],
                     상반기내_통일=t["상반기내_감지율"], 상반기내_기존=P.CUSTOM["상반기내_감지율"],
                     잘못된경보_통일=t["잘못된경보율"], 잘못된경보_기존=P.CUSTOM["잘못된경보율"]))
    sizes[name] = size_table(R, P)
display(pd.DataFrame(rows).set_index("문제"))
pd.concat(sizes)


## 3. 2차 통일 규칙: EPL 규칙 개선 (보정 W1, 채점 W2·W3)

In [ ]:

import epl_rules as E; importlib.reload(E)
a, m, r, wrong = E.pick_epl()
E.U_A_EPL, E.U_M_EPL = a, m
print(f"보정 기간 W1에서 고른 값: α = {a}, m = {m} (W1 시뮬레이션: 좋지 않은 후보 채택 {wrong:.1%})")
rows, reps = [], []
for wn in ["W2 결정 15-19 → 채점 19-23", "W3 결정 18-22 → 채점 22-27"]:
    E.DECIDE, E.TRUTH = E.WINDOWS[wn]; S = E.candidate_stats()
    good = set(S.후보[S.rps_tru_t >= E.GOOD_T]); bad = set(S.후보[S.rps_tru < 0])
    for meth in ["기존 점진법", "빠른-순차+스코어+안전장치", "통일 규칙"]:
        s = E.simulate_window(S, meth, reps=500)
        rows.append(dict(기간=wn, 방식=meth, 맞는채택=s["맞는_채택"], 좋지않은채택=round(1 - s["맞는_채택"] - s["채택_없음"], 3),
                         해로운채택=s["해로운_채택"], 맞는채택까지_시즌=s["맞는_채택까지_시즌"]))
    for meth in ["기존 점진법", "빠른-순차+스코어+안전장치", "통일 규칙"]:
        cur, L = E.replay(meth); n_cum = L.판정까지_경기.cumsum().to_numpy() if len(L) else []
        fg = next((int(n_cum[i]) for i, ch in enumerate(L.바꾼규칙) if ch in good), None) if len(L) else None
        g, lo, hi = E.truth_gain(cur)
        reps.append(dict(기간=wn, 방식=meth, 채택=" → ".join(L.바꾼규칙) or "(없음)", 해로운채택=sum(c in bad for c in L.바꾼규칙),
                         첫좋은채택까지_경기=fg, 채점기간_개선=f"{g:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
display(pd.DataFrame(rows).set_index(["기간", "방식"]))
pd.DataFrame(reps).set_index(["기간", "방식"])



## 4. 결과와 판정 (사전 실행)

**2차 통일 규칙이 고른 값은 세 문제 모두 같았습니다: α = 0.9, m = 0.**

| 문제 | 맞는 판정 (기존 → 통일) | 잘못된 판정 (기존 → 통일) | 조건 1: 잘못된 ≤ 10% | 조건 2: 5%p 이내 |
|---|---|---|---|---|
| 청년 순이동 경보 | 85% → **81%** | 8.2% → **6.3%** | 통과 | 통과 (−3.5%p) |
| 혼인 경보 | 80% → **82%** | 1.1% → **3.7%** | 통과 | 통과 (+2.6%p) |
| EPL 개선 W2 | 83% → **79%** (빠른 방식 대비) | 17% → **21%** | **실패** | 통과 (−4.0%p) |
| EPL 개선 W3 | 95% → **92%** | 5% → **8%** | 통과 | 통과 (−2.8%p) |

- **4개 채점 중 3개 통과.** 실패한 EPL W2는 기존 빠른 방식도 같은 조건에 실패했습니다(17%). 기존 점진법은 51%였습니다.
- **실제 EPL 재생:** 통일 규칙은 W2 210경기, W3 250경기 만에 좋은 후보를 채택했습니다. 빠른 방식보다 20~30경기 빠르고, 해로운 채택은 없었습니다.
- **잃은 것:**
  - 혼인 경보에서 연 3,000건 이상 큰 도시의 잘못된 경보가 5% → 16%로 늘었습니다. 기존 규칙은 큰 도시용 실무 크기를 따로 정했는데, 통일 규칙은 전체 기준 하나만 씁니다.
  - 청년 순이동의 상반기 감지가 51% → 47%로 조금 줄었습니다.

### 알게 된 것
1. **판정 방식은 하나로 통일할 수 있습니다.** 성능 손실은 맞는 판정 기준 최대 4%p였습니다.
2. **세 문제가 같은 값(α = 0.9, m = 0)을 골랐습니다.** 하나의 상수로 고정할 수 있다는 신호입니다.
   - 다만 0.9는 후보 중 가장 민감한 **끝값**이라, 실제 최적값은 더 민감한 쪽에 있을 수 있습니다.
   - "모든 시점에서 오판 5%" 같은 이론적 보장은 사라졌습니다. 남은 것은 **같은 경계 모양 + 같은 보정 규칙**이라는 통일성입니다.
3. **민감도를 고정하면 실패합니다(1차).** 통일해야 하는 것은 숫자가 아니라 **숫자를 정하는 절차**였습니다.

### 다음 검증
- 다음 자료(예: 미분양 경보)에서 **α = 0.9, m = 0을 미리 고정**해 놓고 채점하면, 상수 고정이 맞는지 사전 등록 검증이 됩니다.
- 큰 단위용 실무 크기를 통일 규칙 안에서 어떻게 다룰지(규모별로 m을 정하는 규칙을 하나 더 둘지) 정해야 합니다.
